# Checkpointed 06b sub-01 baseline and matched tri-modal comparison

This notebook first reproduces historical 06b route_c for **sub-01, seed 0** from the full training partition. It performs CPU-only input validation and a CPU update profile before the separately gated GPU run. It never loads official-test EEG, test image features, or the merged train/test InternViT cache. After the valid baseline completes, an optional matched A/B/C/D warm-start run evaluates direct EEG-to-caption retrieval using captions from training images only. No official-test scores are produced.

In [ ]:
# Bootstrap the audited sources before importing torch or any project module.
import hashlib, json, os, pathlib, subprocess, sys
if any(name == 'phase1' or name.startswith('phase1.') for name in sys.modules):
    raise RuntimeError('phase1 is already imported. Restart the runtime before this bootstrap so audited source hashes also identify the code being executed.')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
try:
    from google.colab import drive
except ImportError:
    pass  # Local CPU execution uses the same cells with the environment paths below.
else:
    drive.mount('/content/drive')
BRANCH='research/06b-independent-audit-20261120'
REPO=pathlib.Path(os.environ.get('EEG_06C_REPO', '/content/eeg-alignment-phase1-06c-audit'))
# The branch is a delivery location; these LF-normalized hashes pin the Python sources.
# Regenerate only after reviewing/testing an intentional source change.
EXPECTED_SOURCE_SHA256 = {'phase1/__init__.py': '7c6a15363d636c811994e4a796a59f900f26d697022647aee7a3bb3b6a7fb441',
 'phase1/core.py': 'd7e3c9829cf55d1fcad3465f64ae36ea3342d6812c944ad0a34c613bc77c5cdd',
 'phase1/experiment.py': 'fd2543be8795f934a73ce1a604d8b0f3ca5fa127c43f1108cb2aead3eb448973',
 'phase1/extraction_environment.py': 'ba0fa4f3f4e7c93999d05588d884d44758c696deb099c455a1544c0ad6f580fc',
 'phase1/final_audit.py': '34ec2dc1174b62c161e85bbfac75932a61d2c6de6d22f061774140e07a6f1d93',
 'phase1/historical_06b.py': 'd903470a0b790cd416494fd5e57a5fea6a6243d94e49820610603b2e44de3a28',
 'phase1/input_diagnostics.py': '8169b40154f337cb354c55eb6caa38d4e791599e087d18056575c5a7182ae1cf',
 'phase1/joint_data.py': 'e4acced0c8e238891335261b8551ffe928d0cd7cc026eb8586a711343e1d4630',
 'phase1/joint_experiment.py': '7e7032837ce781ad26c196019f0f18e9d2bb2b8526cb85a704c62452f95e30ce',
 'phase1/joint_inference.py': 'a2a95125d6b5e4fd8abf4b6b427f246515af53153b9386bb6ebe14b2f38e9651',
 'phase1/joint_model.py': 'cf4bbe3c8f0a2c67dbc59bb7ef2cd94fe171cd7a3a2caa875d3889b51f92657c',
 'phase1/joint_smoke.py': 'ca23d172fe66ceb83df5b9b0515448522134c5952fb44725fa0e2fe71ff2f26f',
 'phase1/models.py': 'c080717e68643f57cc78ab952b74d7d026a0857f5ff663133858575a38f56ab9',
 'phase1/patch_adapter.py': '9d5935add9b695982a8a04f9450c84c5ddc2215e8cf81386293538f0506e895a',
 'phase1/patch_cache.py': '163f0d7b565dbdb0f63a671f3c60cc015e2162c67066b26fa06adb8e5c5e5d29',
 'phase1/patch_experiment.py': '0b4e36a687456610c386e43b8114c0e6ca0d2b668890d69373649c9311f94c19',
 'phase1/patch_prepare.py': '5584969828298eb9c1b6c2fcea5bb43f2a7a34afdf416ac59bee3b6147c40e46',
 'phase1/patch_resources.py': '4cd7fda55a14efa4a8c298528a4936d4cfbd0e3265f4982da65fb932fe6ca058',
 'phase1/pilot_inputs.py': 'ea5e1ce03cb3022be701cef1adb97c4601d0c126b25ff685a59d9d7ab96459cb',
 'phase1/pilot_preflight.py': '90155a155c330d61902faed35defeb5343394f7f0e346576c4f8718f54ae2044',
 'phase1/process_log.py': '33da9bfcc7f4c565f792e206894108ffa260bd80e6a275de93ab06d56449a94f',
 'phase1/reconcile.py': '7e561ec8d0bb2fa80f8f986118bdd6a35cfb31aaa3d34e780c8f44a6a09dfa9b',
 'phase1/semantic.py': 'ef6a71155985d499ff9a3f68a1619eb072958477374a50a5f21598f406104aaa',
 'phase1/smoke.py': '75b55d8c8b4ce0de58c3c1ff163bbfa2210384253b401ff0fa5a87393cc897fc',
 'phase1/training_preflight.py': 'b8ab5fffc16aab413338864279942fa04036ecb41b9a39456a93cf75f7e998fe',
 'phase1/trimodal_06b.py': '6fa779798bb8620ff939ca2cb1d2ce51f710ae81ae4119ec7f8bc90af53b07b6',
 'phase1/v7_controls.py': '3ce8d6a8f49db9f3f813c8e3a125fb3eb568e00af5bc7b8628c5de111e9734c8',
 'phase1/validation_root_audit.py': '6d9aa0d693c88132c2a57d1dd5b00f71de2eed6c44114c318009b8769a91ae31'}
def verify_audited_sources(repo):
    actual={path.relative_to(repo).as_posix():hashlib.sha256(path.read_bytes().replace(b'\r\n', b'\n')).hexdigest()
            for path in sorted((repo/'phase1').rglob('*.py'))}
    if actual != EXPECTED_SOURCE_SHA256:
        raise RuntimeError('Audited source hash mismatch. Use the notebook and code from the same verified audit commit.')

if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
else:
    branch=subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
    if branch!=BRANCH or dirty: raise RuntimeError('Existing notebook checkout is on another branch or has local changes; preserve it and use a fresh runtime/path.')
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
actual=subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
if actual != BRANCH: raise RuntimeError(f'Expected {BRANCH}, got {actual}')
verify_audited_sources(REPO)
sys.path.insert(0,str(REPO))
import numpy as np, torch
print('branch:',actual,'commit:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())
print('PyTorch:',torch.__version__,'NumPy:',np.__version__,'CUDA available:',torch.cuda.is_available())


## Required full historical training cache

Default cache root: `/content/drive/MyDrive/eeg_results/cache`. Required: `sub-01_train_z16.npy` with shape `[16540,63,time]`, plus `feat6_internvit6b_parts/train_000.pt` through `train_008.pt`. Each train chunk must contain mean-pooled layers 20/24/28/32/36, width 3200, aligned in the original 06b training image order. The CPU preflight rejects missing, changed, nonfinite, wrong-shape, or wrong-protocol inputs. Hashes and shapes do not establish EEG channel semantics or image-feature row identity; those remain dependent on independent cache provenance. It never falls back to the 2,000-image pilot or opens test files. See `docs/HISTORICAL_06B_CHECKPOINTED_REPRODUCTION.md` for exact requirements and the optional train-only caption artifact contract.

In [ ]:
from phase1.historical_06b import create_manifest, preflight, profile_cpu
CACHE=pathlib.Path(os.environ.get('EEG_CACHE', '/content/drive/MyDrive/eeg_results/cache'))
RUN_ROOT=pathlib.Path(os.environ.get('EEG_06C_RUN_ROOT', '/content/drive/MyDrive/eeg_results/v6b_checkpointed_phase1/sub01_seed0_routec'))
RUN_ROOT.mkdir(parents=True,exist_ok=True)
MANIFEST=RUN_ROOT/'train_only_manifest.json'
if not MANIFEST.exists(): create_manifest(CACHE,MANIFEST)
preflight_report=preflight(MANIFEST)
print(json.dumps(preflight_report,indent=2))
(RUN_ROOT/'cpu_preflight.json').write_text(json.dumps(preflight_report,indent=2))

In [ ]:
# CPU-only representative update; small-batch timing is not a full-schedule estimate.
profile=profile_cpu(MANIFEST,batch_size=32)
print(json.dumps(profile,indent=2))
(RUN_ROOT/'cpu_profile.json').write_text(json.dumps(profile,indent=2))
assert profile['official_test_opened'] is False

## Historical 06b reproduction

The code uses the historical posterior-17 channel order, train-only `z16` EEG unchanged, mean-pooled InternViT layers 20/24/28/32/36, 06b target normalization, original Aligner architecture, route logits `[-2,-1,0,-1,-2]`, AdamW and OneCycleLR (60 epochs, batch 1,024). It selects the best checkpoint using only the historical last-image-per-concept training validation rows. The model state is saved as a plain 06b `Aligner.state_dict()`; epoch-boundary checkpoints support `--resume`.

In [ ]:
from phase1.historical_06b import run_baseline
RUN_BASELINE=False  # Set True only after reviewing the CPU preflight/profile above.
if RUN_BASELINE:
    if not torch.cuda.is_available(): raise RuntimeError('Select a Colab GPU only after CPU preflight passes.')
    baseline_result=run_baseline(MANIFEST,RUN_ROOT/'baseline','cuda',resume=True)
    print(json.dumps({k:v for k,v in baseline_result.items() if k not in ('target_mean','target_std','fit_rows','validation_rows')},indent=2))
else:
    print('GPU training is gated. Set RUN_BASELINE=True after reviewing CPU preflight/profile.')

## Matched tri-modal A/B/C/D (only after the baseline)

This stage refuses to run unless the checkpointed 06b baseline is complete and verified. It additionally requires a separate **training-only** 16,540 × 512 caption embedding array, aligned exact-caption IDs, and a provenance JSON documenting caption generator/encoder/revision, training-only normalization, and `used_test: false`. The mixed legacy caption cache is never opened. All four arms warm-start from identical baseline weights and new-head initialization, use the same historical fit/validation split and training budget, and report image and duplicate-aware text retrieval. Direct text scores use EEG and caption candidates only. If those train-only caption inputs are unavailable, this optional stage skips with no fallback.

In [ ]:
from phase1.trimodal_06b import preflight_text_inputs, run as run_trimodal
TEXT_FEATURES=CACHE/'captions_train_only.npy'
CAPTION_IDS=CACHE/'caption_ids_train_only.json'
CAPTION_PROVENANCE=CACHE/'caption_provenance_train_only.json'
caption_files=[TEXT_FEATURES,CAPTION_IDS,CAPTION_PROVENANCE]
if all(path.is_file() for path in caption_files):
    print(json.dumps(preflight_text_inputs(*caption_files),indent=2))
else:
    print('Tri-modal stage not ready; missing train-only caption inputs:',[str(path) for path in caption_files if not path.is_file()])

In [ ]:
RUN_TRIMODAL=False  # Set True only after baseline completion and train-only caption preflight.
if RUN_TRIMODAL:
    if not torch.cuda.is_available(): raise RuntimeError('Select a Colab GPU only after both CPU preflights pass.')
    if not all(path.is_file() for path in caption_files): raise FileNotFoundError('Train-only caption artifacts are required; mixed caches are not accepted.')
    records=run_trimodal(MANIFEST,RUN_ROOT/'baseline',TEXT_FEATURES,CAPTION_IDS,CAPTION_PROVENANCE,RUN_ROOT/'trimodal_abcd','cuda',resume=True)
    print(json.dumps(records,indent=2))
else:
    print('A/B/C/D GPU training is gated. Set RUN_TRIMODAL=True after baseline and caption preflights pass.')

## Scientific boundary

The historical test evaluation is intentionally absent. Baseline selection and tri-modal reporting use training-partition validation only. The preserved historical image-feature normalization is fitted on all 16,540 training-partition rows, including the 1,654 validation rows; this validation is not isolated from that preprocessing. The new text normalizer and residualizer fit only optimizer rows. Cache hashes and tensor shapes prove file stability and structural compatibility, not image-row identity or the original EEG channel order. The original 06b result and this development validation are not an untouched-test comparison. This notebook does not claim SOTA, significance, or a tri-modal gain. Preserve each `run.json`, input hash report, CPU profile, epoch log, and checkpoint with the Drive output.